In [1]:
# Kiểm tra tính nhất quán của dữ liệu ngày mà 
# lịch sử đều đồng bộ với node loc

# Thư viện

In [2]:
import osmnx as ox

import pandas as pd
import numpy as np

from pathlib import Path

import requests
import json

from rapidfuzz import fuzz
from unidecode import unidecode

# Load các file csv

In [3]:
PATH = Path("../data/raw")

files = {
    "nodes": PATH / "nodes.csv",
    "streets": PATH / "streets.csv",
    "segments": PATH / "segments.csv",
    "status": PATH / "segment_status.csv",
    "train": PATH / "train.csv"
}

In [4]:
nodes_df = pd.read_csv(files["nodes"])
streets_df = pd.read_csv(files["streets"])
segments_df = pd.read_csv(files["segments"])
status_df = pd.read_csv(files["status"])
train_df = pd.read_csv(files["train"])

In [5]:
osm_ways_full_df = pd.read_csv("../data/preprocess/osm_ways_full.csv")

C:\Users\Asriel\AppData\Local\Temp\ipykernel_3704\878787935.py:1: DtypeWarning: Columns (2,4,8,9,10,18,20,32,37,39,40,41,43,44,46,47,48,49,50,51,52,53,54,55,61,62,63,64,65,66,67,72,73,74,77,79,80,81,82,83,84,85,86,88,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,120,121,122,123,124,125,126,129,130,131,132,133,134,136,137,138,139,140,142,143,145,146,149,150,152) have mixed types. Specify dtype option on import or set low_memory=False.
  osm_ways_full_df = pd.read_csv("../data/preprocess/osm_ways_full.csv")


In [6]:
pre_proc_df = pd.read_csv("../data/preprocess/all.csv")

In [7]:
pre_proc_df.head()

,_id,segment_id,date,weekday,period,LOS,s_node_id,e_node_id,length,street_id,...,long_snode,lat_snode,long_enode,lat_enode,name,street_type,status_id,updated_at,velocity,time_bucket
0,0,26,2021-04-16,4,period_0_30,A,366428456,366416066,0.116,32575820,...,106.768732,10.841506,106.769254,10.842422,Nguyễn Văn Bá,tertiary,89252,2021-04-16 00:55:31.333000+00:00,117,2021-04-16 00:30:00+00:00
1,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,892,2020-08-02 23:56:28.019000+00:00,34,2020-08-02 23:30:00+00:00
2,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,4501,2020-08-02 23:59:28.137000+00:00,26,2020-08-02 23:30:00+00:00
3,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,8110,2020-08-03 00:02:28.070000+00:00,2,2020-08-03 00:00:00+00:00
4,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,11719,2020-08-03 00:05:28.204000+00:00,35,2020-08-03 00:00:00+00:00


In [8]:
train_nodes = list(set(train_df["s_node_id"]) | set(train_df["e_node_id"]))

In [9]:
train_streets = set(train_df["street_id"].to_list())

# Load graph

In [10]:
with open("../data/raw/osm_train_2019_01_03.json", "r", encoding="utf-8") as f:
    osm_data = json.load(f)

In [11]:
osm_data.keys()

dict_keys(['version', 'generator', 'osm3s', 'elements'])

In [12]:
osm_data["version"]

0.6

## OSM Element

In [13]:
osm_elements = pd.DataFrame(osm_data["elements"])

In [14]:
osm_elements.head()

,type,id,lat,lon,tags,nodes,members
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN


In [15]:
osm_elements["type"].unique()

array(['node', 'way', 'relation'], dtype=object)

## OSM Node

In [16]:
osm_nodes_df = osm_elements[osm_elements["type"] == "node"]

In [17]:
osm_nodes_df.head()

,type,id,lat,lon,tags,nodes,members
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN


## OSM Way

In [18]:
osm_way_df = osm_elements[osm_elements["type"] == "way"]

In [19]:
print(osm_way_df.shape)
osm_way_df.head()

(7606, 7)


,type,id,lat,lon,tags,nodes,members
3061,way,32575768,NaN,NaN,"{'name': 'Đường số 27', 'highway': 'residential'}","[366369613, 5795144851, 366418963, 366373068, ...",NaN
3062,way,32576350,NaN,NaN,"{'name': 'Đường số 18', 'highway': 'residential'}","[366372346, 5755079612, 3040292134, 5755079614...",NaN
3063,way,32576691,NaN,NaN,"{'addr:city': 'Ho Chi Minh', 'addr:district': ...","[366452320, 3351962143, 3351962141, 366375776,...",NaN
3064,way,32576911,NaN,NaN,"{'highway': 'residential', 'name': 'Tân Thành'}","[5778381635, 5552002921, 4878713065, 580288880...",NaN
3065,way,32577060,NaN,NaN,{'highway': 'residential'},"[366371080, 366389438, 5735589498, 5735589492,...",NaN


In [20]:
data = []

for _, row in osm_way_df.iterrows():
    way_id = row['id']
    nodes = row['nodes']
    if isinstance(nodes, list) and len(nodes) >= 2:
        for i in range(len(nodes)-1):
            data.append({
                'way_id': way_id,
                's_node_id': nodes[i],      # from
                'e_node_id': nodes[i+1]     # to
            })

osm_edges_df = pd.DataFrame(data)
print(osm_edges_df.head())

     way_id   s_node_id   e_node_id
0  32575768   366369613  5795144851
1  32575768  5795144851   366418963
2  32575768   366418963   366373068
3  32575768   366373068  5753228946
4  32575768  5753228946  5795144815


In [21]:
osm_reverse_edges_df = osm_edges_df.rename(columns={
    "s_node_id":"e_node_id", 
    "e_node_id":"s_node_id"
})
osm_undirected_edges_df = pd.concat([osm_edges_df, osm_reverse_edges_df])
print(osm_undirected_edges_df.shape)
osm_undirected_edges_df.head()

(112706, 3)


,way_id,s_node_id,e_node_id
0,32575768,366369613,5795144851
1,32575768,5795144851,366418963
2,32575768,366418963,366373068
3,32575768,366373068,5753228946
4,32575768,5753228946,5795144815


In [22]:
osm_way_tags_df = pd.json_normalize(
    osm_way_df["tags"]
        .where(
            osm_way_df["tags"]
                .notna(), 
            other=[{}]
        )
)

osm_way_tags_df.insert(0, "way_id", osm_way_df["id"].to_numpy())
osm_way_tags_df.head()

,way_id,name,highway,addr:city,addr:district,name:en,ref,addr:subdistrict,service,oneway,...,name:ja,name:th,fixme,traffic_signals,lay,information,covered,motorcar:forward,crossing,footway
0,32575768,Đường số 27,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,32576350,Đường số 18,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32576691,Phan Văn Hớn,secondary,Ho Chi Minh,Hoc Mon,Phan Van Hon,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,32576911,Tân Thành,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,32577060,NaN,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Kiểm tra tính nhất quán

## Node

### Node id

In [23]:
osm_nodes = set(osm_nodes_df["id"])
train_nodes = set(train_df["s_node_id"]) | set(train_df["e_node_id"])

In [24]:
inter_train_osm_nodes = osm_nodes & train_nodes

In [25]:
print(len(osm_nodes))
print(len(train_nodes))
print(len(inter_train_osm_nodes))

53115
11314
11314


### Node location

In [26]:
osm_node_locs = set(osm_nodes_df[["id", "lon", "lat"]].apply(tuple, axis=1))
train_node_locs = set(train_df[["s_node_id", "long_snode", "lat_snode"]].apply(tuple, axis=1)) | \
                    set(train_df[["e_node_id", "long_enode", "lat_enode"]].apply(tuple, axis=1))

In [27]:
inter_osm_train_node_locs = osm_node_locs & train_node_locs
print(len(osm_node_locs))
print(len(train_node_locs))
print(len(inter_osm_train_node_locs))

53115
11314
11314


## Way

### Way id

In [28]:
train_segment_ids = set(train_df["street_id"])
osm_ways_ids = set(osm_way_df["id"])
inter_osm_train_edge = osm_ways_ids & train_segment_ids

print(len(train_segment_ids))
print(len(osm_ways_ids))
print(len(inter_osm_train_edge))

1967
7606
1967


### Way nodes

In [29]:
train_segment_nodes = set(train_df[["street_id", "s_node_id", "e_node_id"]].apply(tuple, axis=1))
osm_way_nodes = set(osm_undirected_edges_df.apply(tuple, axis=1))
inter_osm_train_edges = osm_way_nodes & train_segment_nodes

print(len(train_segment_nodes))
print(len(osm_way_nodes))
print(len(inter_osm_train_edges))

10027
112706
10027


## Way type

In [30]:
pre_proc_df[["street_id", "street_type"]]

,street_id,street_type
0,32575820,tertiary
1,32575862,secondary
2,32575862,secondary
3,32575862,secondary
4,32575862,secondary
...,...,...
90933,654864528,primary
90934,654864530,tertiary
90935,654864530,tertiary
90936,654864530,tertiary


In [31]:
train_type_df = pre_proc_df[["street_id", "street_type"]]
osm_type_df = osm_ways_full_df[["id", "tags.highway"]]

In [32]:
train_type = set(train_type_df.apply(tuple, axis=1))
osm_type = set(osm_type_df.apply(tuple, axis=1))
inter_osm_train_type = train_type & osm_type

print(len(train_type))
print(len(osm_type))
print(len(inter_osm_train_type))

1967
8330
1580


In [33]:
combine_type_df = train_type_df.merge(
    osm_type_df,
    how="inner",
    left_on="street_id",
    right_on="id"
).drop_duplicates()

combine_type_df = combine_type_df[combine_type_df["street_type"] != combine_type_df["tags.highway"]]

print(combine_type_df.shape)
combine_type_df.head()

(206, 4)


,street_id,street_type,id,tags.highway
33,32575935,primary_link,32575935,trunk_link
60,32576099,unclassified,32576099,tertiary
63,32576112,unclassified,32576112,tertiary
195,32576820,tertiary,32576820,residential
202,32577095,tertiary,32577095,secondary


In [34]:
combine_type_df.head(20)

,street_id,street_type,id,tags.highway
33,32575935,primary_link,32575935,trunk_link
60,32576099,unclassified,32576099,tertiary
63,32576112,unclassified,32576112,tertiary
195,32576820,tertiary,32576820,residential
202,32577095,tertiary,32577095,secondary
261,32577138,unclassified,32577138,tertiary
1559,32577684,secondary,32577684,primary
5608,32579421,unclassified,32579421,residential
7624,32580466,tertiary,32580466,residential
7894,32580469,unclassified,32580469,tertiary


In [35]:
street_types = pre_proc_df["street_type"].unique()
segment_types = pre_proc_df["segment_type"].unique()
osm_highways = combine_type_df["tags.highway"].unique()

In [36]:
print(osm_highways)

['trunk_link' 'tertiary' 'residential' 'secondary' 'primary'
 'primary_link' 'trunk' 'secondary_link' 'service' 'unclassified'
 'tertiary_link']


In [37]:
segment_types

array(['tertiary', 'secondary', 'unclassified', 'primary_link', 'primary',
       'trunk', 'trunk_link', 'residential', 'secondary_link',
       'tertiary_link', 'motorway_link', 'pitch', 'motorway',
       'convenience', 'university', 'car', 'house', 'company', 'school',
       'marketplace', 'fuel', 'clothes', 'government', 'bus_station',
       'bank', 'cinema'], dtype=object)

In [38]:
street_types

array(['tertiary', 'secondary', 'unclassified', 'primary_link', 'primary',
       'trunk', 'trunk_link', 'secondary_link', 'tertiary_link',
       'motorway_link', 'motorway'], dtype=object)

In [39]:
# Tập hợp tất cả tag unique
all_tags = sorted(set(street_types) | set(segment_types) | set(osm_highways))

# Tạo DataFrame với tag làm index
mapping_df = pd.DataFrame(index=all_tags)

# Điền dữ liệu
mapping_df['street_type'] = [tag if tag in street_types else np.nan for tag in all_tags]
mapping_df['segment_type'] = [tag if tag in segment_types else np.nan for tag in all_tags]
mapping_df['osm_highway'] = [tag if tag in osm_highways else np.nan for tag in all_tags]

# Đổi tên index cho rõ ràng
mapping_df.index.name = 'tag'

print(mapping_df.shape)

(27, 3)


In [40]:
mapping_df

,street_type,segment_type,osm_highway
tag,,,
bank,NaN,bank,NaN
bus_station,NaN,bus_station,NaN
car,NaN,car,NaN
cinema,NaN,cinema,NaN
clothes,NaN,clothes,NaN
company,NaN,company,NaN
convenience,NaN,convenience,NaN
fuel,NaN,fuel,NaN
government,NaN,government,NaN
